# Assignment 4

In [1]:
import pandas as pd

### Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:

### - Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account", "general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g. if d%3 gives "account", write a question like "how do I update my registered mobile number").
#### - \# Example roll number ...23 -> digits 2, 3
#### - \# digit 2 -> category[2 % 3] = general
#### - \# digit 3 -> category[3 % 3] = billing

### Output: Print your final 6-row DataFrame.

In [2]:
roll = "1024170173"
digits = [int(c) for c in roll[-2:]]
cats = ["billing", "account", "general"]
for d in digits:
    print(d, "->", cats[d % 3])

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

custom_entries = [
    {"question": "how do i update my registered mobile number",
     "answer": "Go to Account Settings > Profile > Edit Mobile Number and verify with the OTP.",
     "keywords": "mobile number update", "category": "account"},
    {"question": "why was my late payment fee charged",
     "answer": "A late fee is added if payment is made after the due date on your invoice.",
     "keywords": "late fee penalty", "category": "billing"},
]

df = pd.DataFrame(fixed_entries + custom_entries)
print(df)

7 -> account
3 -> billing
                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5          why was my late payment fee charged   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3         You can pay via UPI, card, or net banking.     pay payment upi fee   
4  Go to Account Settings > Profile > Edit Mobile...    mobile number update   
5  A late fee is added if payment is made after t...        late fee penalty   

  category  
0  billing  
1  account  
2  general  
3  billing

### Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.

In [3]:
def score_query(query, df):
    words = set(query.lower().split())
    results = []
    for _, row in df.iterrows():
        entry_words = set(row["question"].lower().split()) | set(row["keywords"].lower().split())
        score = len(words & entry_words)
        if score > 0:
            results.append({"question": row["question"], "answer": row["answer"],
                            "category": row["category"], "confidence": round(score / len(words), 2)})
    results.sort(key=lambda x: x["confidence"], reverse=True)
    return pd.DataFrame(results)

score_query("how do i pay my fee", df)

,question,answer,category,confidence
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,0.67
1,how do i update my registered mobile number,Go to Account Settings > Profile > Edit Mobile...,account,0.67
2,why was my late payment fee charged,A late fee is added if payment is made after t...,billing,0.33
3,what is the annual fee,The annual fee is Rs 500.,billing,0.17
4,how to reset password,Go to Settings > Reset Password.,account,0.17


### Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [4]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]

print(same_category("account", df))

1                          how to reset password
4    how do i update my registered mobile number
Name: question, dtype: str


### Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named 1024170173_faq_data.csv.

In [5]:
new_keyword = input("Enter a new keyword: ").strip().lower()
df.at[0, "keywords"] += " " + new_keyword
df.to_csv(roll + "_faq_data.csv", index=False)
print(df.at[0, "keywords"])

fee cost price charge annual


### Q5: Using groupby, print how many FAQ entries you have per category.

In [6]:
print(df.groupby("category").size())

category
account    2
billing    3
general    1
dtype: int64


Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one - it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [7]:
def score_query(query, df):
    words = set(query.lower().split())
    results = []
    for _, row in df.iterrows():
        entry_words = set(row["question"].lower().split()) | set(row["keywords"].lower().split())
        score = len(words & entry_words)
        if score > 0:
            results.append({"question": row["question"], "answer": row["answer"],
                            "category": row["category"], "confidence": round(score / len(words), 2)})
    results.sort(key=lambda x: x["confidence"], reverse=True)
    res = pd.DataFrame(results)
    if res.empty:
        print("No match found")
        return res
    top = res[res["confidence"] == res["confidence"].max()]
    if len(top) > 1:
        print("Tie between", len(top), "entries:")
    else:
        print("Best match:")
    for _, r in top.iterrows():
        print("-", r["question"], "|", r["answer"])
    return res

In [8]:
_ = score_query("fee", df)

Tie between 3 entries:
- what is the annual fee | The annual fee is Rs 500.
- how can i pay the fee | You can pay via UPI, card, or net banking.
- why was my late payment fee charged | A late fee is added if payment is made after the due date on your invoice.


In [9]:
_ = score_query("reset password", df)

Best match:
- how to reset password | Go to Settings > Reset Password.
